[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_02_data_structures_algorithms/ch08_tuples_sets_dicts/03_coding_exam_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 8 Hands-On Coding Exam — Structured & Hash-Based Data Audit (100 Points)

**Book Reference:** Chapter 8, Section 8.7 (Hands-On Coding Exam)  
**Suggested Time:** 35–45 minutes (Open Book, Closed Solution)

| Task | Public Health / Biostatistics Function | Points |
| :--- | :--- | :---: |
| **Task 8.1** | `audit_survey_ids(enrolled_ids, completed_ids)` (Set deduplication & audit dict) | 30 |
| **Task 8.2** | `summarize_district_anemia(records)` (Sentinel filtering & per-district prevalence) | 35 |
| **Task 8.3** | `merge_clinic_inventories(warehouse_a, warehouse_b)` (Non-mutating sum merge) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Exam Auto-Grader
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import sharukh_tutor as tutor

tutor.activate(chapter="ch08", notebook="coding_exam")


---
## Exam Task 8.1 (30 Points): Survey ID Completion Auditor

Write `audit_survey_ids(enrolled_ids, completed_ids)` to audit longitudinal field survey completion:
1. Deduplicate `enrolled_ids` into a set (`enrolled_unique` = number of unique enrolled IDs).
2. Count **valid** completions as IDs present in **both** `enrolled_ids` and `completed_ids` (ignoring duplicates and any stray unregistered IDs in `completed_ids`).
3. Compute `missing_ids` as the **set** of enrolled IDs who have not yet completed the survey.
4. Compute `completion_pct` as `(completed_valid / enrolled_unique) * 100` rounded to **1 decimal place** (or `0.0` if `enrolled_unique == 0`).
5. Return the dictionary with exact keys: `"enrolled_unique"`, `"completed_valid"`, `"missing_ids"`, and `"completion_pct"`.

In [ ]:
def audit_survey_ids(enrolled_ids, completed_ids):
    """Audit unique enrollment, valid completion, missing IDs, and pct."""
    enrolled_set = set(enrolled_ids)
    valid_completed = enrolled_set & set(completed_ids)
    missing_ids = enrolled_set - valid_completed
    enrolled_unique = len(enrolled_set)
    completed_valid = len(valid_completed)
    if enrolled_unique == 0:
        completion_pct = 0.0
    else:
        completion_pct = round((completed_valid / enrolled_unique) * 100, 1)
    return {
        "enrolled_unique": enrolled_unique,
        "completed_valid": completed_valid,
        "missing_ids": missing_ids,
        "completion_pct": completion_pct,
    }


---
## Exam Task 8.2 (35 Points): District-Level Maternal Anemia Summarizer

Write `summarize_district_anemia(records)` accepting a list of dictionaries with keys `"district"` and `"hb_g_dl"`:
1. Skip any record where `hb_g_dl` is `None` or sentinel `999`.
2. Classify a valid reading as anemic if `hb_g_dl < 11.0` (strictly `< 11.0`; `11.0` is Normal).
3. Return a dictionary mapping each district (with at least 1 valid reading) to its anemia prevalence percentage `(anemic / valid) * 100` rounded to **1 decimal place**.
4. Omit districts that have **zero** valid readings after sentinel filtering.

In [ ]:
def summarize_district_anemia(records):
    """Return {district: anemia_prevalence_pct} skipping None and 999."""
    valid_counts = {}
    anemic_counts = {}
    for rec in records:
        hb = rec["hb_g_dl"]
        if hb is None or hb == 999:
            continue
        dist = rec["district"]
        valid_counts[dist] = valid_counts.get(dist, 0) + 1
        if hb < 11.0:
            anemic_counts[dist] = anemic_counts.get(dist, 0) + 1
    return {
        dist: round((anemic_counts.get(dist, 0) / n) * 100, 1)
        for dist, n in valid_counts.items()
    }


---
## Exam Task 8.3 (35 Points): Non-Mutating Clinic Inventory Merger

Write `merge_clinic_inventories(warehouse_a, warehouse_b)` accepting two dictionaries mapping supply names to integer quantities:
1. Return a **brand-new** dictionary containing every supply from both warehouses, **summing** quantities for any shared keys.
2. Do **NOT** mutate either `warehouse_a` or `warehouse_b`, and do **NOT** return a direct reference/alias to either input dictionary.

In [ ]:
def merge_clinic_inventories(warehouse_a, warehouse_b):
    """Return a new dict summing shared supply keys without mutating inputs."""
    combined = dict(warehouse_a)
    for item, qty in warehouse_b.items():
        combined[item] = combined.get(item, 0) + qty
    return combined


In [ ]:
exam_submissions = {
    "ch08_exam_01": audit_survey_ids,
    "ch08_exam_02": summarize_district_anemia,
    "ch08_exam_03": merge_clinic_inventories,
}

tutor.check_exam("ch08", exam_submissions)


---
## Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 8.1 (Stray IDs & Empty Guard):** Counting `len(set(completed_ids))` directly includes unregistered stray IDs (e.g., `"P99"`), which can push completion over `100%`! Intersecting `enrolled_set & set(completed_ids)` guarantees only legitimate enrollees count, and guarding `enrolled_unique == 0` avoids `ZeroDivisionError`.
- **Task 8.2 (Order of Sentinel Check & Strict `< 11.0`):** Checking `hb < 11.0` before `hb is None` triggers `TypeError`; using `<= 11.0` misclassifies normal `11.0 g/dL` readings as anemic. Accumulating counts only for valid readings naturally omits all-invalid districts.
- **Task 8.3 (Aliasing vs. Copying & Overwriting vs. Summing):** Writing `combined = warehouse_a` creates a shared alias that mutates the caller's dictionary, and writing `{**warehouse_a, **warehouse_b}` overwrites overlapping keys instead of summing them. Copying with `dict(warehouse_a)` and adding with `combined.get(item, 0) + qty` solves both bugs.